In [1]:
from tensorly_custom.tucker_tensor import validate_tucker_rank, tucker_normalize, TuckerTensor
from tensorly_custom.decomposition._tucker import tucker_to_tensor
from tensorly_custom.base import unfold
from tensorly_custom.tenalg import mode_dot
import tensorly_custom as tl
import pytensorlab as ptl
import numpy as np
from typing import List, Tuple
from utils import DATA_DIR, select_gpu, tree_to_device, notify_discord
from tucker_tensor import SparseTupleTensor
from typing import Optional, Union
import cupy as cp
import cupyx.scipy.sparse as cpx_sparse
import time
import torch

from tqdm import tqdm


# ----- cupy sparse methods -----

def fro_norm_coo(x):
    # x: cupyx.scipy.sparse.coo_matrix
    data = x.data
    return cp.sqrt((cp.abs(data) ** 2).sum())

def unfold_from_vectorized_sparse(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape,
    mode: int,
    to_dense: bool = False,
):
    """
    Unfold a sparse tensor that is stored as a vectorized CuPy sparse matrix.

    Parameters
    ----------
    vec_tensor : cupyx.scipy.sparse.spmatrix
        Sparse matrix of shape (np.prod(orig_shape), 1) created by
        `torch_sparse_to_cupy` for an N-D tensor.
    orig_shape : tuple[int, ...]
        Original N-D tensor shape, e.g. (I0, I1, I2).
    mode : int
        Mode along which to unfold.
    to_dense : bool, default False
        If True, return a dense cupy.ndarray.
        If False, return a cupy sparse COO matrix.

    Returns
    -------
    unfolded : cupy.ndarray or cupyx.scipy.sparse.coo_matrix
        Mode-`mode` unfolding of shape
        (orig_shape[mode], np.prod(orig_shape) // orig_shape[mode]).
    """
    # Make sure we're in COO format

    cu = vec_tensor.tocoo()

    row_cp = cu.row
    col_cp = cu.col
    data_cp = cu.data

    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    # ---- move to host and use int64 for safe arithmetic ----
    row_np = cp.asnumpy(row_cp).astype(np.int64)
    col_np = cp.asnumpy(col_cp).astype(np.int64)

    flat_np = row_np + col_np * np.int64(block_size)

    coords = np.unravel_index(flat_np, orig_shape)

    row_unf_np = coords[mode]

    other_coords = coords[:mode] + coords[mode + 1:]
    other_shape = tuple(s for i, s in enumerate(orig_shape) if i != mode)
    col_unf_np = np.ravel_multi_index(other_coords, other_shape)

    row_unf_cp = cp.asarray(row_unf_np)
    col_unf_cp = cp.asarray(col_unf_np)

    unfolded_shape = (orig_shape[mode], int(np.prod(other_shape)))
    unfolded = cpx_sparse.coo_matrix(
        (data_cp, (row_unf_cp, col_unf_cp)),
        shape=unfolded_shape,
    )

    if to_dense:
        return unfolded.toarray()
    return unfolded


def left_dense_mul_sparse(
    mat: cp.ndarray,
    sp: cpx_sparse.spmatrix
) -> Union[cp.ndarray, cpx_sparse.coo_matrix]:
    """
    Compute mat @ sp, choosing dense or sparse output based on a simple
    memory heuristic.

    mat: cupy ndarray of shape (R, I_mode)
    sp:  cupy sparse matrix of shape (I_mode, K)
    """
    sp = sp.tocoo()
    R, I_mode = mat.shape
    assert I_mode == sp.shape[0], f"mat shape {mat.shape} not compatible with sparse {sp.shape}"

    # Let CuPy handle dense @ sparse; result is cupy.ndarray
    return mat @ sp
def fold_unfolded_sparse_to_vec(
    unfolded: Union[cpx_sparse.spmatrix, cp.ndarray],
    old_shape: Tuple[int, ...],
    mode: int,
    new_dim: int,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Fold a mode-`mode` unfolded matrix back to a vectorized sparse tensor.

    unfolded:
        - sparse COO or any cupyx.scipy.sparse.spmatrix of shape (new_dim, prod(other_dims)), or
        - dense cupy.ndarray of the same shape.
    old_shape : original N-D shape BEFORE replacing dimension at `mode`
    mode      : mode index that was unfolded
    new_dim   : new size at `mode` (typically rank[mode])

    Returns
    -------
    vec_sparse : COO of shape (prod(new_shape), 1)
    new_shape  : tuple of ints, updated tensor shape
    """

    old_shape = tuple(old_shape)
    N = len(old_shape)

    new_shape = list(old_shape)
    new_shape[mode] = new_dim
    new_shape = tuple(new_shape)

    other_shape = tuple(s for i, s in enumerate(old_shape) if i != mode)

    if cpx_sparse.isspmatrix(unfolded):
        unfolded = unfolded.tocoo()
        row = unfolded.row
        col = unfolded.col
        data = unfolded.data
    else:
        row, col = cp.nonzero(unfolded)
        data = unfolded[row, col]

    coords_other = cp.unravel_index(col, other_shape)

    coords_full = []
    idx_other = 0
    for i in range(N):
        if i == mode:
            coords_full.append(row)
        else:
            coords_full.append(coords_other[idx_other])
            idx_other += 1

    coords_full = tuple(coords_full)

    size = int(np.prod(new_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    flat = cp.ravel_multi_index(coords_full, new_shape)

    # --- block encoding of flat indices ---
    row_vec = flat % block_size
    col_vec = flat // block_size

    n_blocks = int((size + block_size - 1) // block_size)
    vec_sparse = cpx_sparse.coo_matrix(
        (data, (row_vec, col_vec)),
        shape=(block_size, n_blocks),
    )
    vec_sparse.sum_duplicates()

    return vec_sparse, new_shape


def sparse_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    curr_shape: Tuple[int, ...],
    factor: cp.ndarray,
    mode: int,
    transpose_factor: bool = True,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Perform a mode-`mode` product on a vectorized sparse tensor (prod(curr_shape), 1),
    using a dense factor matrix, and return the new vectorized sparse tensor.

    vec_tensor: sparse COO (prod(curr_shape), 1)
    curr_shape: current tensor shape
    factor:     dense matrix of shape (I_mode, R_mode) (or R_mode, I_mode if transpose_factor=False)
    mode:       mode index in [0, len(curr_shape))
    transpose_factor: if True, use factor.T (for Tucker-style X ×_n W_n^T)

    Returns
    -------
    new_vec:   sparse COO (prod(new_shape), 1)
    new_shape: updated shape, with dimension at `mode` replaced by R_mode
    """
    curr_shape = tuple(curr_shape)
    I_mode = curr_shape[mode]

    # Factor handling
    if transpose_factor:
        # factor is (I_mode, R_mode) => mat is (R_mode, I_mode)
        assert factor.shape[0] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = tl.transpose(factor)  # (R_mode, I_mode)
    else:
        # factor is already (R_mode, I_mode)
        assert factor.shape[1] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = factor

    R_mode = mat.shape[0]

    # 1) Unfold current sparse tensor along this mode (sparse COO)
    unfolded = unfold_from_vectorized_sparse(
        vec_tensor,
        curr_shape,
        mode,
        to_dense=False,
    )  # shape: (I_mode, prod(other_dims))

    # 2) Left-multiply with dense matrix; currently returns dense cp.ndarray
    #    -> shape: (R_mode, prod(other_dims))
    unfolded_new = left_dense_mul_sparse(mat, unfolded)

    # 3) Fold back into a new vectorized sparse tensor with updated shape
    new_vec, new_shape = fold_unfolded_sparse_to_vec(
        unfolded_new,
        old_shape=curr_shape,
        mode=mode,
        new_dim=R_mode,
    )
    return new_vec, new_shape
def sparse_multi_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape: Tuple[int, ...],
    factors: List[cp.ndarray],
    modes: Optional[List[int]] = None,
    transpose_factors: bool = True,
) -> cp.ndarray:
    """
    multi_mode_dot for a vectorized sparse tensor (prod(orig_shape), 1),
    applying dense factor matrices along the given modes, **staying sparse**
    until the final (small) result, which is densified.

    vec_tensor: sparse COO (prod(orig_shape), 1)
    orig_shape: original tensor shape
    factors:    list of factor matrices, one per mode index
                factor[n] has shape (I_n, R_n)
    modes:      list of modes to apply; if None, uses range(len(factors))
    transpose_factors: if True, uses factors[n].T (Tucker-style)
    """
    if modes is None:
        modes = list(range(len(factors)))

    current_vec = vec_tensor
    current_shape = tuple(orig_shape)

    # Apply each mode in any order (commutes)
    for mode in modes:
        # print("Applying mode", mode)
        current_vec, current_shape = sparse_mode_dot_vec(
            current_vec,
            current_shape,
            factors[mode],
            mode=mode,
            transpose_factor=transpose_factors,
        )

    # At this point, current_vec is still sparse (prod(core_shape), 1)
    core_shape = current_shape  # typically your (50, 50, 50) or similar
    # should not overflow the cupy 32bit index limit if dimensions stay reasonable
    # Finally densify the small core
    coo = current_vec.tocoo()
    flat = coo.row
    data = coo.data

    # Build dense core
    coords = cp.unravel_index(flat, core_shape)
    core_dense = cp.zeros(core_shape, dtype=data.dtype)
    core_dense[coords] = data

    return core_dense
def build_sparse_ratio_vec(
    vec_tensor: cpx_sparse.spmatrix,
    shape,
    core,
    factors,
    epsilon: float = 1e-12,
) -> cpx_sparse.coo_matrix:
    """Build a sparse vectorized tensor Y with the same structure as X,
    but data = X_i / R_i, where R is the current reconstruction.

    vec_tensor : vectorized sparse X (block-encoded)
    shape      : original N-D shape
    core       : current core G
    factors    : list of A^{(n)}
    """
    shape = tuple(shape)
    size = int(np.prod(shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    # 1) Dense reconstruction R = G ×_1 A^{(1)} × ... ×_N A^{(N)}
    #    (follows the paper literally; for huge tensors this may be too big
    #     and you’d want a custom "R at nonzeros only" kernel instead.)
    R = tucker_to_tensor((core, factors))        # dense cp.ndarray of shape `shape`
    R = tl.clip(R, a_min=epsilon, a_max=None)
    R_flat = R.ravel()                           # length = size

    # 2) Same block encoding as elsewhere
    coo = vec_tensor.tocoo()
    row_cp = coo.row
    col_cp = coo.col
    data_cp = coo.data

    # move indices to host to do int64 arithmetic safely
    row_np = cp.asnumpy(row_cp).astype(np.int64)
    col_np = cp.asnumpy(col_cp).astype(np.int64)

    flat_np = row_np + col_np * np.int64(block_size)  # flat indices into R_flat
    flat_cp = cp.asarray(flat_np)

    # 3) Compute X_i / R_i at the same positions
    R_vals = R_flat[flat_cp]                          # cp.ndarray
    R_vals = tl.clip(R_vals, a_min=epsilon, a_max=None)

    ratio_data = data_cp / R_vals                     # X_i / R_i

    # 4) Build sparse ratio vec with identical row, col, shape
    vec_ratio = cpx_sparse.coo_matrix(
        (ratio_data, (row_cp, col_cp)),
        shape=vec_tensor.shape,
    )
    #vec_ratio.sum_duplicates()  # just in case

    return vec_ratio

def ptl_tucker_to_tensor(tucker: ptl.TuckerTensor,
                         skip_factor: Optional[int] = None) -> np.ndarray:
    """Reconstruct full tensor from Tucker representation, optionally skipping one factor."""
    factors = tucker.factors
    if skip_factor is not None:
        factors = [f for i, f in enumerate(factors) if i != skip_factor]
    return ptl.tmprod(tucker.core, factors, list(range(tucker.ndim)) if skip_factor is None else
                     [i for i in range(tucker.ndim) if i != skip_factor])

#
# def sparse_dense_division_batched(X_csr_gpu, R_host, batch_rows=10_000):
#     """
#     X_csr_gpu : cupyx.scipy.sparse.csr_matrix on GPU
#     R_host    : numpy.ndarray on CPU with same shape as X
#     batch_rows: number of rows to process per batch
#     """
#     N, M = X_csr_gpu.shape
#     assert R_host.shape == (N, M)
#
#     # Store GPU sparse chunks and vstack at end
#     batches = []
#     for start in tqdm(range(0, N, batch_rows)):
#         stop = min(start + batch_rows, N)
#         # Sparse slice is still on GPU (cheap)
#         X_batch_gpu = X_csr_gpu[start:stop]
#
#         # Move matching dense slice to GPU
#         R_batch_gpu = cp.asarray(R_host[start:stop])
#
#         # Elementwise division only for nonzeros (sparse × dense)
#         # multiply is implemented efficiently on sparse
#         Y_batch_gpu = X_batch_gpu.multiply(1.0 / R_batch_gpu)
#
#         batches.append(Y_batch_gpu)
#
#     # Combine back into one sparse matrix on GPU
#     Y_gpu = cpx_sparse.vstack(batches, format='csr')
#     return Y_gpu

def gather_dense_at_block_nz(dense_nd: np.ndarray,
                             vec_tensor: cpx_sparse.spmatrix,
                             orig_shape) -> cp.ndarray:
    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    dense_flat = dense_nd.reshape(size, order="C")
    print(type(dense_flat), dense_flat.shape)
    coo = vec_tensor.tocoo()
    flat = coo.row + coo.col * block_size
    print(type(flat), flat.shape)
    return dense_flat[flat.get()]

#
# def dense_tensor_to_block_matrix(x, orig_shape, xp=cp):
#     """
#     x: dense N-D tensor (numpy or cupy)
#     orig_shape: the N-D shape you used for ravel_multi_index/unravel_index
#     returns: dense block-matrix of shape (block_size, n_blocks)
#     """
#     size = int(np.prod(orig_shape))
#     int32_max = np.iinfo(np.int32).max
#     block_size = min(size, int32_max)
#     n_blocks = int((size + block_size - 1) // block_size)
#
#     flat = xp.asarray(x).reshape(orig_shape).ravel(order="C")  # MUST be C-order
#     padded_len = block_size * n_blocks
#     if flat.size < padded_len:
#         flat = xp.pad(flat, (0, padded_len - flat.size))
#
#     # Columns are consecutive blocks => reshape with Fortran order
#     block_mat = flat.reshape((block_size, n_blocks), order="F")
#     return block_mat


def print_elapsed_time(start_time, message=""):
    """Prints the elapsed time since start_time."""
    now = time.time()
    # if the message starts with indents (tabs), add the same number to the elapsed time print
    tabs = ""
    for char in message:
        if char == "\t":
            tabs += "\t"
        else:
            break
    elapsed_time = now - start_time
    minutes, seconds = divmod(elapsed_time, 60)
    if message:
        print(message)
    print(f"{tabs}Elapsed time: {int(minutes)} minutes and {seconds} seconds.")
    return now



2025-12-15 13:36:07.088828: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
device = select_gpu(3)
tl.set_backend("cupy")

In [3]:
dims = 3000
sparse_tensor = SparseTupleTensor.load_from_disk(
    dataset="fineweb_large",
    method="sc",
    dims=dims,
    map_location="cpu"
)
sparse_tensor.tensor_to_sparse("sparse")
ptl_tensor = ptl.SparseTensor(data=sparse_tensor.tensor.data,
                        indices=sparse_tensor.tensor.coords,
                        shape=sparse_tensor.tensor.shape)
sparse_tensor.tensor_to_sparse("torch")
sparse_tensor.tensor_to_sparse("cupy")
n_iter_max=10
init="random"
tol=10e-5
random_state=42
verbose=False
return_errors="full"
normalize_factors=False
patience: int=3
rank = [100, 100, 100]
shape = tuple(sparse_tensor.shape)
tensor = sparse_tensor.tensor
rank = validate_tucker_rank(shape, rank=rank)
epsilon = 1e-12
modes = list(range(len(rank)))
non_negative = True
# skip_factor = None
# transpose_factors = False
max_cuda_mem = torch.cuda.get_device_properties().total_memory
batch_rows = max_cuda_mem // (dims ** 2 * 12)

if init == "random":
    rng = tl.check_random_state(random_state)
    core = tl.tensor(
        rng.random_sample([rank[index] for index in range(len(modes))]) + 0.01,
        **tl.context(tensor),
    )
    factors = [
        tl.tensor(
            rng.random_sample((shape[mode], rank[index])), # we changed this to original shape
            **tl.context(tensor),
        )
        for index, mode in enumerate(modes)
    ]
else:
    (core, factors) = init

if non_negative is True:
    factors = [tl.abs(f) for f in factors]
    core = tl.abs(core)
else:
    raise NotImplementedError("Currently only non-negative=True is supported.")

In [4]:
start = time.time()
for mode_n in modes:
    X = unfold_from_vectorized_sparse(tensor, shape, mode_n)

    # Z cannot be computed in high dimensions
    Z = tucker_to_tensor((core, factors), skip_factor=mode_n)
    ptl_Z = ptl_tucker_to_tensor(
        ptl.TuckerTensor(core=cp.asnumpy(core), factors=[cp.asnumpy(f) for f in factors]),
        skip_factor=mode_n
    )
    # Verify that Z from both implementations are close
    # Z_cpu = cp.asnumpy(Z)
    # assert np.allclose(Z_cpu, ptl_Z), "Z tensors from different implementations do not match!"
    Z = unfold(Z, mode_n) # (50, 1000000)
    A = factors[mode_n] # (1000, 50)

    rows = X.row        # (nnz,)
    cols = X.col        # (nnz,)
    vals = X.data       # (nnz,)

    R_nz = cp.zeros_like(vals)  # (nnz,)

    for r in range(rank[0]):          # R = rank[mode_n], e.g. 50
        A_r = A[:, r]           # (I,)
        Z_r = Z[r, :]           # (K,)

        # Gather A[i,r] for all nonzeros and Z[r,j] for all nonzeros:
        A_r_vals = A_r[rows]    # (nnz,)
        Z_r_vals = Z_r[cols]    # (nnz,)

        R_nz += A_r_vals * Z_r_vals  # elementwise, stays (nnz,)
    # print("R:", type(R_nz), R_nz.shape)
    R_nz = tl.clip(R_nz, a_min=epsilon, a_max=None)
    w = vals / R_nz            # (nnz,)
    w = w[:, cp.newaxis]       # (nnz, 1)

    W_data = (vals / R_nz)     # (nnz,)
    W = cpx_sparse.coo_matrix(
        (W_data, (rows, cols)),
        shape=X.shape,         # (I_n, K)
    )
    # we transform to csr for faster row-wise operations later
    W = W.tocsr()
    # print(f"W shape: {W.shape}, Z shape {Z.shape}")
    # print(f"W type: {type(W)}, Z type {type(Z)}")
    numerator = W @ tl.transpose(Z)          # (I_n, R)
    # # this breaks.
    # R = tl.dot(A, Z) # (1000, 1000000)
    # Z_T = tl.transpose(Z) # (1000000, 50)
    # frac = X / R # (1000, 100000)
    # numerator = frac @ Z_T # (1000, 50)

    # Denominator: E Z^T  ==  row-wise broadcast of sum over columns of Z
    den_row = tl.sum(Z, axis=1)    # (R,)
    denominator = den_row[np.newaxis, :]   # (1, R) – will broadcast to (I, R)
    # print(f"Numerator shape: {numerator.shape}, Denominator shape: {denominator.shape}")
    # print(f"numerator type: {type(numerator)}, denominator type: {type(denominator)}")
    # Multiplicative update
    A = A * (numerator / (denominator + 1e-12))

    A = tl.clip(A, a_min=epsilon, a_max=None)

    factors[mode_n] = A


a = print_elapsed_time(start, "KL factors calculated:")

KL factors calculated:
Elapsed time: 0 minutes and 5.3716065883636475 seconds.


# Core calculations


In [ ]:
import os, multiprocessing
from threadpoolctl import threadpool_limits  # NEW

max_cpu_frac = 0.8
def get_eval_num_threads(fraction: float = 0.75, min_threads: int = 1) -> int:
    """Return n_threads ≈ fraction * available CPUs (at least min_threads)."""
    try:
        n_cores = multiprocessing.cpu_count()
    except NotImplementedError:
        n_cores = os.cpu_count() or 1

    n_threads = max(min_threads, int(n_cores * fraction))
    return n_threads

n_threads = get_eval_num_threads(fraction=max_cpu_frac, min_threads=1)

core_np = tl.to_numpy(core)
factors_np = [tl.to_numpy(f) for f in factors]
start = time.time()
ptl_tucker = ptl.TuckerTensor(core=core_np,
                            factors=factors_np)
ptl_time = print_elapsed_time(start, "Converted to PyTensorLab Tucker:")
with threadpool_limits(n_threads):
    ptl_R = ptl_tucker_to_tensor(ptl_tucker)
R_time = print_elapsed_time(ptl_time, "Reconstructed R:")
data = gather_dense_at_block_nz(ptl_R, tensor, shape)
data_time = print_elapsed_time(R_time, "KL factors calculated:")
data = tl.clip(data, a_min=epsilon, a_max=None)
X_R_time = print_elapsed_time(data_time, "Gathered R at X nonzeros:")
X_R_data = tensor.data / cp.asarray(data)
div_time = print_elapsed_time(X_R_time, "Computed X / R at nonzeros:")

Converted to PyTensorLab Tucker:
Elapsed time: 0 minutes and 0.013541698455810547 seconds.


In [13]:
ptl_X_R = cpx_sparse.coo_matrix(
    (X_R_data, (tensor.row, tensor.col)),
    shape=tensor.shape,
)
X_R = sparse_multi_mode_dot_vec(
    vec_tensor=ptl_X_R,
    orig_shape=shape,
    factors=factors,
    modes=modes,
    transpose_factors=True,  # X ×_n W_n^T
)
X_R = tl.clip(X_R, a_min=epsilon, a_max=None)

col_sums = [tl.sum(A_n, axis=0) for A_n in factors]
F = col_sums[0].reshape((core.shape[0],) + (1,) * (core.ndim - 1))
for n in range(1, core.ndim):
    shape_n = [1] * n + [core.shape[n]] + [1] * (core.ndim - n - 1)
    F = F * col_sums[n].reshape(tuple(shape_n))
F = tl.clip(F, a_min=epsilon, a_max=None)
core *= X_R / (F + epsilon)

core, factors = tucker_normalize((core, factors))

# Divergence calculation

In [ ]:
core_np = tl.to_numpy(core)
factors_np = [tl.to_numpy(f) for f in factors]
start = time.time()
ptl_tucker = ptl.TuckerTensor(core=core_np,
                              factors=factors_np)
with threadpool_limits(n_threads):
    ptl_R = ptl_tucker_to_tensor(ptl_tucker)
vec_tensor = tensor
epsilon = 1e-12

shape = tuple(shape)
size = int(np.prod(shape))
int32_max = np.iinfo(np.int32).max
block_size = min(size, int32_max)

# --- 1) Dense reconstruction R = G ×_1 A^{(1)} × ... ×_N A^{(N)} ---
# This is exactly step 3 in Table 2 of the paper.
# R = tucker_to_tensor((core, factors))      # cp.ndarray, shape=shape
# R = tl.clip(R, a_min=epsilon, a_max=None)
# R_flat = R.ravel()                         # length = size

r_nz = gather_dense_at_block_nz(ptl_R, tensor, shape)
r_nz = tl.clip(data, a_min=epsilon, a_max=None)

# --- 2) Decode sparse X indices to flat indices ---
X_coo = vec_tensor.tocoo()
row_cp = X_coo.row
col_cp = X_coo.col
x_nz = X_coo.data

# same block-encoding as in unfold_from_vectorized_sparse
row_np = cp.asnumpy(row_cp).astype(np.int64)
col_np = cp.asnumpy(col_cp).astype(np.int64)
flat_np = row_np + col_np * np.int64(block_size)
flat_cp = cp.asarray(flat_np)

# --- 3) X_i and R_i at nonzero entries ---
# r_nz = R_flat[flat_cp]                    # R_i where X_i ≠ 0
# r_nz = tl.clip(r_nz, a_min=epsilon, a_max=None)
# the original data can still contain harmful zeros
x_nz = tl.clip(x_nz, a_min=epsilon, a_max=None)

print(f"x_nz: {type(x_nz)}, r_nz: {type(r_nz)}")
r_nz = cp.asarray(r_nz)

# --- 4) KL contribution from nonzeros ---
# sum_{i: X_i>0} [X_i log(X_i/R_i) - X_i + R_i]
term_pos = x_nz * cp.log(x_nz / r_nz) - x_nz + r_nz
kl_pos = cp.sum(term_pos)

# --- 5) KL contribution from zero entries ---
# For X_i = 0, the KL term tends to R_i (limit X→0).
# Full sum over all i is:
#   ∑_i [X_i log(X_i/R_i) - X_i + R_i]
# = (sum over nonzeros) + (sum over zeros),
# and sum_{zeros} R_i = sum(R) - sum_{nonzeros} R_i.
sum_R = cp.sum(ptl_R)
sum_R_nz = cp.sum(r_nz)
kl_zero = sum_R - sum_R_nz
kl_total = kl_pos + kl_zero

# --- 6) Optional normalized "relative" KL error ---
sum_X = cp.sum(x_nz)  # sum over nonzero X
rel_kl = kl_total / cp.maximum(sum_X, epsilon)
print(f"KL divergence: {kl_total}, relative KL: {rel_kl}")